# 🧩 Feature Extraction from Messy Text Data — Study Notes

**Topic:** Data Cleaning & Feature Engineering with Pandas
**Dataset:** `anime.csv` (scraped anime ranking data)
**Skill level:** Beginner → Intermediate

---

### 📌 What is this notebook about?

When data is **scraped from a website**, it often lands in a single messy text
column instead of neat, separate columns. In this notebook, the `Title` column
of an anime dataset looks like this:

```
Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr 2009 - Jul 20103,218,472 membersManga StoreVolume 1€4.58Preview
```

Buried inside that one string are **several useful features**:
- 🎬 Episode count → `64`
- 📅 Air date range → `Apr 2009 - Jul 2010`
- ⏳ How long the show ran for → `16 months`

**Feature extraction** is the process of pulling structured, usable columns
out of raw/unstructured data like this — a core skill in real-world data
science, since clean datasets are rare in practice.

> 💡 **Big picture:** This notebook is a great example of *string parsing +
> pandas `apply()`* — a pattern you will use constantly when cleaning
> scraped or user-generated data.

## 1️⃣ Setup — Importing Libraries

In [2]:
import numpy as numpy   # numerical operations (used indirectly via pandas)
import pandas as pd     # core library for reading, cleaning, and analyzing tabular data

- **pandas (`pd`)** → the main tool for loading and manipulating tables (DataFrames).
- **numpy** → pandas is built on top of numpy, so it's commonly imported alongside it.

> ⚠️ **Note:** The conventional alias for numpy is `np`, not `numpy`
> (i.e. `import numpy as np`). The code here works fine either way — Python
> doesn't care what alias you use — but `as np` is the widely recognized
> convention, and interviewers / reviewers will expect it.

## 2️⃣ Loading the Dataset

In [3]:
# loading the data
df = pd.read_csv(r'anime.csv')   # r'...' = raw string, useful for Windows-style file paths

- `pd.read_csv()` reads a CSV file into a **DataFrame** (pandas' table structure).
- The `r` before the string (`r'anime.csv'`) marks it as a **raw string** —
  this matters mainly on Windows paths like `r'C:\Users\data\anime.csv'`,
  where backslashes would otherwise be treated as escape characters
  (e.g. `\n` = newline). For a simple filename like this it has no visible
  effect, but it's a good habit for file paths in general.

> 💡 **Tip:** This assumes `anime.csv` is in the same folder as the notebook.
> If you get a `FileNotFoundError`, check your working directory with
> `import os; os.getcwd()`.

## 3️⃣ Exploring the Raw Data

In [4]:
df.head()

,Rank,Title,Score
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05


Look closely at the **`Title`** column above — it isn't just a title. It's a
single string that concatenates *several* pieces of information scraped
straight from the webpage:

```
Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr 2009 - Jul 20103,218,472 members
   ↑ title                ↑ episodes    ↑ air date range      ↑ member count
```

This is a very common real-world scenario: the scraper grabbed a whole HTML
block of text without separating the fields. **Our job in this notebook is
to split that mess into clean, usable columns.**

### 🎯 Objectives for This Notebook

- [x] Make a new column for **episode count**
- [x] Make a new column for **timestamp** (air date range)
- [x] Which anime has the **highest score**?
- [ ] Give the **top 5** highest scoring anime
- [ ] Which anime has the **highest episode count**?
- [ ] Animes with the **top 5 episode counts**
- [ ] Which is the **longest running anime**?

> 📝 The checked items are implemented below. The unchecked ones are natural
> **practice exercises** — see the recap section at the end of the notebook
> for hints on how to tackle them using what you've learned here.

## 4️⃣ Feature 1 — Extracting Episode Count

**Goal:** Pull the number of episodes (e.g. `64`) out of text like
`TV (64 eps)Apr 2009 - Jul 2010...`.

**Pattern to notice:** the episode count always sits **inside parentheses**
`( ... )`, right before the word `eps`.

In [5]:
df.loc[2]['Title']  # peek at one raw title to confirm the pattern

'Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 - Dec 2022474,138 members'

- `.loc[2]` selects the row with **index label** `2` (here, same as position
  since the index is the default 0,1,2,...).
- Confirms the pattern: `(13 eps)` sits between parentheses, exactly like
  the earlier example.

In [6]:
def extract_episodes(txt):
    check = False        # flag: True once we're "inside" the parentheses
    data = ""             # accumulator for characters found inside ( )
    for i in txt:                 # scan the string one character at a time
        if i == ')':               # stop as soon as the closing bracket is hit
            break
        if check == True:          # if we're inside the parentheses, keep the char
            data = data + i
        if i == '(':                # once we see '(', start collecting from the NEXT char
           check = True
    return data

**How this function works, step by step:**

1. Walk through the string **one character at a time**.
2. When it sees `'('`, it flips `check` to `True` (meaning: "start collecting now").
3. While `check` is `True`, every character gets appended to `data`.
4. As soon as it hits `')'`, the loop **breaks** and returns everything
   collected so far — this text looks like `"64 eps"`.

> 🧠 **Trace example:** for `"TV (64 eps)Apr..."`
> - Sees `T`, `V`, ` ` → ignored (check is still `False`)
> - Sees `(` → `check = True` (nothing appended yet)
> - Sees `6`, `4`, ` `, `e`, `p`, `s` → each appended → `data = "64 eps"`
> - Sees `)` → loop breaks, returns `"64 eps"`

> 💡 **Interview tip:** This is a classic **manual string-parsing** approach
> (O(n) single pass). In practice, this exact task is usually done more
> concisely with a **regular expression**, e.g.:
> ```python
> import re
> df['Title'].str.extract(r'\((\d+)\s*eps\)')
> ```
> Knowing *both* approaches is valuable — the manual loop shows you understand
> the underlying logic, while regex shows you can write production-efficient
> code. Interviewers often ask you to solve it manually first, then optimize.

In [7]:
df['Episodes'] = df['Title'].apply(extract_episodes)  # run the function on every row

- `.apply()` runs `extract_episodes` **once per row** on the `Title` column
  and stores the results in a brand-new `Episodes` column.

> ⚠️ **Performance note:** `.apply()` with a Python function loops row-by-row
> under the hood, so it's slower than pandas' built-in **vectorized** string
> methods (like `.str.extract()`) on large datasets. Fine here since the
> dataset is small — but worth knowing for bigger data or coding interviews
> about performance.

In [8]:
df

,Rank,Title,Score,Episodes
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10,64 eps
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07,24 eps
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06,13 eps
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06,51 eps
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05,10 eps
5,6,"Gintama'TV (51 eps)Apr 2011 - Mar 2012534,105 ...",9.04,51 eps
6,7,Gintama: The FinalMovie (1 eps)Jan 2021 - Jan ...,9.04,1 eps
7,8,Hunter x Hunter TV (148 eps)Oct 2011 - Sep 201...,9.04,148 eps
8,9,Kaguya-sama wa Kokurasetai: Ultra RomanticTV (...,9.04,13 eps
9,10,Gintama': EnchousenTV (13 eps)Oct 2012 - Mar 2...,9.03,13 eps


The new `Episodes` column now exists — but notice it still contains the text `"64 eps"`, not a clean number. We'll fix that next.

## 5️⃣ Cleaning the Episodes Column

In [9]:
df["Episodes"] = df['Episodes'].str.replace(" eps", "")  # strip the " eps" suffix, leaving just digits

- `.str.replace(" eps", "")` is a **vectorized string method** — it applies
  the replacement across the *entire column at once*, without an explicit
  Python loop. This is the more efficient sibling of the manual
  character-by-character approach used above.

In [10]:
df

,Rank,Title,Score,Episodes
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10,64
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07,24
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06,13
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06,51
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05,10
5,6,"Gintama'TV (51 eps)Apr 2011 - Mar 2012534,105 ...",9.04,51
6,7,Gintama: The FinalMovie (1 eps)Jan 2021 - Jan ...,9.04,1
7,8,Hunter x Hunter TV (148 eps)Oct 2011 - Sep 201...,9.04,148
8,9,Kaguya-sama wa Kokurasetai: Ultra RomanticTV (...,9.04,13
9,10,Gintama': EnchousenTV (13 eps)Oct 2012 - Mar 2...,9.03,13


In [11]:
df.loc[0]["Episodes"]  # check a single value

'64'

The output is `'64'` — note the **quotes**. That means it's still a
**string**, not a number, even though " eps" is gone. Pandas won't let us do
numeric operations (like `.mean()` or `.max()`) on it yet.

In [12]:
df['Episodes'] = df['Episodes'].astype(int)  # convert the string column to integers

In [13]:
df

,Rank,Title,Score,Episodes
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10,64
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07,24
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06,13
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06,51
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05,10
5,6,"Gintama'TV (51 eps)Apr 2011 - Mar 2012534,105 ...",9.04,51
6,7,Gintama: The FinalMovie (1 eps)Jan 2021 - Jan ...,9.04,1
7,8,Hunter x Hunter TV (148 eps)Oct 2011 - Sep 201...,9.04,148
8,9,Kaguya-sama wa Kokurasetai: Ultra RomanticTV (...,9.04,13
9,10,Gintama': EnchousenTV (13 eps)Oct 2012 - Mar 2...,9.03,13


In [14]:
type(df.loc[0]['Episodes'])  # confirm the dtype conversion worked

numpy.int64

> 💡 **Interview tip:** The type shown is `numpy.int64`, not Python's built-in
> `int`. Pandas stores numeric columns using **numpy's fixed-size integer
> types** for memory efficiency and speed. `numpy.int64` behaves like a
> normal integer in almost all everyday code, but this distinction
> ("pandas/numpy dtypes vs. native Python types") is a common interview
> question.

### ✅ Section Takeaway — Episodes Feature
- Extracted text between `(` and `)` using a manual character scan.
- Cleaned the `" eps"` suffix with a vectorized `.str.replace()`.
- Converted the cleaned string to a proper numeric type with `.astype(int)`.
- **Pattern learned:** *extract → clean → convert type* — this 3-step
  pipeline is extremely common in real-world data cleaning.

## 6️⃣ Feature 2 — Extracting the Timestamp (Air Date Range)

**Goal:** Pull the air date range (e.g. `Apr 2009 - Jul 2010`) out of the
`Title` text.

**Pattern to notice:** the date range starts **right after the closing
parenthesis** `)` that ends the episode count.

In [19]:
df.loc[0]['Title']  ## timestap starting after baracket close

'Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr 2009 - Jul 20103,218,472 membersManga StoreVolume 1€4.58Preview'

Confirms it: right after `(64 eps)` comes `Apr 2009 - Jul 2010`, followed immediately by the member count with no separator.

In [22]:
def extraction_time(txt):
    data = ""
    for i in range(len(txt)):        # scan character positions by index
        if txt[i] == ')':             # found the closing bracket of "(NN eps)"
            for j in range(i + 1, i + 20):   # grab the next 20 characters after it
                data += txt[j]

            return data                # stop after the first ')' is processed

**How this function works:**

1. Loop through the string **by index** (`i`), so we can look ahead.
2. The moment we find `')'`, we know the date range starts right after it.
3. Grab the **next 20 characters** (a fixed-size window) and return them
   immediately — the function exits on the very first `')'` it finds.

> ⚠️ **Common mistakes / limitations to be aware of (great interview talking
> points):**
> - **Magic number (`20`):** the window size is hardcoded. It happens to work
>   here because `"Apr 2009 - Jul 2010"` is close to 20 characters, but it's
>   fragile — a differently formatted date could get cut short or include
>   trailing junk.
> - **No bounds checking:** if `')'` appeared very close to the end of the
>   string (fewer than 20 characters remaining), `txt[j]` would raise an
>   `IndexError`. This code happens to be safe *for this dataset*, but isn't
>   generally robust.
> - **Fixed-width assumption:** this only works because every date range in
>   this dataset happens to be a similar length. A regex like
>   `r'\)([A-Za-z]{3} \d{4} - [A-Za-z]{3} \d{4})'` would be more robust
>   since it matches the *actual pattern* instead of guessing a length.

In [26]:
df['Timestamp'] = df['Title'].apply(extraction_time)  # run the function on every row

In [28]:
df

,Rank,Title,Score,Episodes,Timestamp
0,1,Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...,9.10,64,Apr 2009 - Jul 2010
1,2,"Steins;GateTV (24 eps)Apr 2011 - Sep 20112,473...",9.07,24,Apr 2011 - Sep 2011
2,3,Bleach: Sennen Kessen-henTV (13 eps)Oct 2022 -...,9.06,13,Oct 2022 - Dec 2022
3,4,"Gintama°TV (51 eps)Apr 2015 - Mar 2016605,113 ...",9.06,51,Apr 2015 - Mar 2016
4,5,Shingeki no Kyojin Season 3 Part 2TV (10 eps)A...,9.05,10,Apr 2019 - Jul 2019
5,6,"Gintama'TV (51 eps)Apr 2011 - Mar 2012534,105 ...",9.04,51,Apr 2011 - Mar 2012
6,7,Gintama: The FinalMovie (1 eps)Jan 2021 - Jan ...,9.04,1,Jan 2021 - Jan 2021
7,8,Hunter x Hunter TV (148 eps)Oct 2011 - Sep 201...,9.04,148,Oct 2011 - Sep 2014
8,9,Kaguya-sama wa Kokurasetai: Ultra RomanticTV (...,9.04,13,Apr 2022 - Jun 2022
9,10,Gintama': EnchousenTV (13 eps)Oct 2012 - Mar 2...,9.03,13,Oct 2012 - Mar 2013


In [31]:
type(df.loc[0]['Timestamp'])  # confirm this is a plain Python string

str

Unlike `Episodes`, this stays a plain Python `str` — no numeric conversion needed yet, since a date *range* isn't a single number. That's handled next.

## 7️⃣ Feature 3 — Calculating How Many Months the Anime Ran

**Goal:** Turn `"Apr 2009 - Jul 2010"` into a single number: total months
the show aired for.

In [34]:
from dateutil.relativedelta import relativedelta   # handles calendar-aware date math (e.g. months, years)
from datetime import datetime                       # for parsing date strings

def calculate_total_months(period):
    try:
        start_str, end_str = period.split(' - ')             # split "Apr 2009 - Jul 2010" into two halves
        start_date = datetime.strptime(start_str, '%b %Y')   # parse "Apr 2009" -> a datetime object
        end_date = datetime.strptime(end_str, '%b %Y')       # parse "Jul 2010" -> a datetime object
        r = relativedelta(end_date, start_date)               # difference expressed in years/months
        return r.years * 12 + r.months + 1  # +1 to include the starting month
    except:
        return None   # if the text doesn't match the expected format, skip it gracefully

df['Months'] = df['Timestamp'].apply(calculate_total_months)

**Breaking this down:**

- `from dateutil.relativedelta import relativedelta` — `dateutil` is a
  **third-party library** (install with `pip install python-dateutil` if
  it's missing). `relativedelta` calculates differences in *calendar units*
  (years/months), unlike `datetime.timedelta`, which only works in raw days.
- `period.split(' - ')` — splits `"Apr 2009 - Jul 2010"` into
  `["Apr 2009", "Jul 2010"]` using the `" - "` separator.
- `datetime.strptime(start_str, '%b %Y')` — parses text into a real date
  object. The **format code** `%b %Y` means:
  - `%b` → abbreviated month name (`Jan`, `Feb`, ... `Dec`)
  - `%Y` → 4-digit year (`2009`)
- `relativedelta(end_date, start_date)` — gives the difference as
  `.years` and `.months` separately (e.g. 1 year, 3 months) instead of a
  raw day count.
- `r.years * 12 + r.months + 1` — converts years→months and adds them to the
  leftover months, then adds `1` so that the **starting month itself counts**
  (e.g. Apr→Jul is 3 full month-gaps, but the show technically aired across
  4 calendar months: Apr, May, Jun, Jul).
- `try/except` — if a title doesn't match the expected `"Mon YYYY - Mon YYYY"`
  format (e.g. ongoing shows, malformed text), the function safely returns
  `None` instead of crashing the whole `.apply()` call.

> 💡 **Interview tip:** `try/except` inside a function used with `.apply()`
> is a very common and useful pattern — it lets you process an entire
> column even when a few rows have "dirty" or unexpected data, without
> halting the whole pipeline.

### ✅ Section Takeaway — Timestamp & Duration Features
- Extracted a fixed-width text window after the `)` character.
- Parsed date strings using `datetime.strptime()` with format codes.
- Computed a calendar-aware difference using `relativedelta`.
- Wrapped the parsing logic in `try/except` for resilience against bad data.

## 8️⃣ Quick Data Exploration — Scores

With clean features in place, let's answer one of our original questions:
**which anime has the highest score?**

In [43]:
df[df['Score'] == df['Score'].max()]['Title']  # filter rows where Score equals the max Score

0    Fullmetal Alchemist: BrotherhoodTV (64 eps)Apr...
Name: Title, dtype: object

This uses **boolean masking**, a core pandas pattern:
1. `df['Score'] == df['Score'].max()` creates a column of `True`/`False`
   values (`True` only for the row(s) with the highest score).
2. `df[ ... ]` uses that mask to filter the DataFrame down to matching rows.
3. `['Title']` then selects just the `Title` column from that filtered result.

> 💡 **Tip:** This pattern — `df[df[col] == condition]` — is one of the most
> frequently used idioms in pandas and comes up constantly in interviews.

In [40]:
df['Score'].value_counts()  # count how many times each score value appears

Score
9.04    4
8.79    3
8.71    2
8.77    2
8.80    2
8.81    2
8.82    2
8.83    2
8.84    2
8.88    2
8.75    2
8.91    2
8.93    2
8.94    2
8.99    2
9.03    2
9.06    2
8.78    2
8.76    1
8.74    1
8.73    1
8.72    1
9.10    1
9.07    1
8.89    1
8.98    1
9.02    1
9.05    1
8.87    1
Name: count, dtype: int64

`.value_counts()` counts occurrences of each unique value and, by default, sorts the result in **descending order of frequency** (most common score first).

In [42]:
df['Score'].unique  # ⚠️ see the note below about this line

<bound method Series.unique of 0     9.10
1     9.07
2     9.06
3     9.06
4     9.05
5     9.04
6     9.04
7     9.04
8     9.04
9     9.03
10    9.03
11    9.02
12    8.99
13    8.99
14    8.98
15    8.94
16    8.94
17    8.93
18    8.93
19    8.91
20    8.91
21    8.89
22    8.88
23    8.88
24    8.87
25    8.84
26    8.84
27    8.83
28    8.83
29    8.82
30    8.82
31    8.81
32    8.81
33    8.80
34    8.80
35    8.79
36    8.79
37    8.79
38    8.78
39    8.78
40    8.77
41    8.77
42    8.76
43    8.75
44    8.75
45    8.74
46    8.73
47    8.72
48    8.71
49    8.71
Name: Score, dtype: float64>

> ⚠️ **Common mistake spotted!** `.unique` is written here **without
> parentheses**, so instead of *calling* the method, this returns a
> `<bound method Series.unique of ...>` object — a reference to the method
> itself, not the actual array of unique values.
>
> **The fix** is simply to call it as a function:
> ```python
> df['Score'].unique()   # returns an array of unique score values
> ```
> This is a very easy mistake to make (and to miss!) — always double check
> that methods are being *called* with `()` when you expect a result rather
> than a method reference. It's a classic beginner bug that also shows up in
> technical interviews as a "spot the bug" question.

## 9️⃣ Objectives Recap — What's Done vs. Left as Practice

| Objective | Status | Notes |
|---|---|---|
| New column for episode count | ✅ Done | `Episodes` column, cleaned & converted to `int` |
| New column for timestamp | ✅ Done | `Timestamp` column, plus derived `Months` duration |
| Highest scoring anime | ✅ Done | via boolean masking on `Score` |
| Top 5 highest scoring anime | 📝 Exercise | Try `df.sort_values('Score', ascending=False).head(5)` |
| Highest episode count anime | 📝 Exercise | Try `df[df['Episodes'] == df['Episodes'].max()]` |
| Top 5 by episode count | 📝 Exercise | Try `df.sort_values('Episodes', ascending=False).head(5)` |
| Longest running anime | 📝 Exercise | Use the `Months` column you already built: `df.sort_values('Months', ascending=False).head(1)` |

> 🧠 Notice that once `Episodes`, `Timestamp`, and `Months` were extracted as
> **proper numeric columns**, answering the remaining questions becomes a
> single line of pandas code (`sort_values`, `max`, `head`) — this is the
> whole point of feature extraction: *turn messy text into columns you can
> directly query.*

## 📚 Summary — Key Takeaways

1. **Scraped data is messy** — a single text field often hides multiple
   useful features that need to be manually separated.
2. **The extract → clean → convert pipeline** is the backbone of feature
   engineering:
   - *Extract* the relevant substring (manual loop or regex).
   - *Clean* it (strip unwanted text, e.g. `.str.replace()`).
   - *Convert* it to the right type (`.astype(int)`, `datetime`, etc.).
3. **Manual string parsing vs. vectorized/regex methods** — loops make the
   logic explicit and easy to learn from, but pandas' built-in vectorized
   string methods (`.str.extract()`, `.str.replace()`) and regex are faster
   and more robust for production code.
4. **`datetime.strptime` + `relativedelta`** is a reliable combo for turning
   text date ranges into meaningful numeric durations.
5. **`try/except` inside `.apply()`** keeps a pipeline running smoothly even
   when some rows don't match the expected format.
6. **Boolean masking** (`df[df[col] == value]`) is the standard way to filter
   rows based on a condition.

---

## 🎤 Interview Tips (Consolidated)

- Be ready to explain **why** you'd choose regex/vectorized operations over
  a manual character loop (performance on large datasets) — but also be able
  to *write* the manual loop, since it demonstrates understanding of the
  underlying logic.
- Know the difference between `datetime.timedelta` (raw day/second
  differences) and `dateutil.relativedelta` (calendar-aware year/month/day
  differences).
- Understand `numpy.int64` vs Python's built-in `int` — pandas uses numpy
  dtypes internally for performance.
- Be able to explain `.loc[]` (label-based indexing) vs positional indexing.
- Practice reading pandas boolean-mask filtering (`df[df[col] == x]`) fluently
  — it's one of the most common patterns tested in data-focused interviews.

## 🐛 Common Mistakes to Avoid (Consolidated)

- Forgetting `()` when calling a method (e.g. `.unique` vs `.unique()`).
- Hardcoding "magic numbers" (like the `20`-character window) instead of
  matching the actual pattern with regex.
- Not handling `IndexError`/format mismatches when slicing strings by a
  fixed length.
- Forgetting to **convert** a cleaned text column to a numeric/date type
  before doing numeric operations on it (a string `'64'` won't `.mean()`).
- Using `import numpy as numpy` instead of the conventional `as np` (works,
  but not idiomatic — may cause confusion when reading others' code).